# Уравнение Пуассона. Схема МКО из лекции 5

**Постановка и сетка (слайд 3).** Решаем $u''=f(x)=-\pi^2\sin(\pi x)$ на $[0,1]$ с условиями $u_L=0$, $u_R=2$. При $N=5$ отрезок состоит из пяти объёмов $V_i=[(i-1)h,ih]$, $h=1/5$. Искомое $u_i$ относится к центру $x_i=(i-1/2)h$.

**Баланс и источник (слайды 4–6).** Интегрирование по $V_i$ даёт $j_{i+1/2}-j_{i-1/2}=h f_i$, где $j=u'$ и $f_i$ — среднее значение источника. Для заданного синуса вычисляем его точно:
$$f_i=\frac1h\int_{(i-1)h}^{ih}f(x)\,dx
=\frac{\pi}{h}\big[\cos(\pi i h)-\cos(\pi(i-1)h)\big].$$

**Потоки (слайд 7).** На общей грани $j_{i+1/2}\approx(u_{i+1}-u_i)/h$. На внешних гранях $j_{1/2}\approx(u_1-u_L)/(h/2)$ и $j_{N+1/2}\approx(u_R-u_N)/(h/2)$.

**Система (слайды 8–9).** После подстановки потоков и умножения баланса на $h$ получаем
$$\begin{aligned}
-3u_1+u_2 &=h^2 f_1-2u_L,\\
u_{i-1}-2u_i+u_{i+1}&=h^2 f_i,\quad 2\le i\le N-1,\\
u_{N-1}-3u_N&=h^2 f_N-2u_R .
\end{aligned}$$

In [ ]:
import numpy as np

Lx = 1.0
u_left, u_right = 0.0, 2.0

def u_ref(x):
    return np.sin(np.pi * x) + 2 * x

def assemble_system(n):
    h = Lx / n
    faces = np.linspace(0, Lx, n + 1)
    x = (faces[:-1] + faces[1:]) / 2
    f_mean = np.pi * np.diff(np.cos(np.pi * faces)) / h

    A = np.diag(np.full(n, -2.0))
    A += np.diag(np.ones(n - 1), 1) + np.diag(np.ones(n - 1), -1)
    A[0, 0] -= 1
    A[-1, -1] -= 1
    b = h**2 * f_mean
    b[0] -= 2 * u_left
    b[-1] -= 2 * u_right
    return x, A, b

def solve_poisson(dx):
    if not np.isfinite(dx) or dx <= 0:
        raise ValueError("Шаг должен быть конечным и положительным.")
    x, A, b = assemble_system(max(1, round(Lx / dx)))
    return x, np.linalg.solve(A, b)

In [ ]:
x, A, b = assemble_system(5)
u = np.linalg.solve(A, b)
print("Матрица A для пяти объёмов\n", A)
print("Правая часть b\n", b)
print(" i     x_i        u_i      u точное     |ошибка|")
for i, (xi, ui) in enumerate(zip(x, u), start=1):
    exact = u_ref(xi)
    print(f"{i:2d}    {xi:.1f}    {ui:.6f}    {exact:.6f}    {abs(ui-exact):.6f}")

In [ ]:
import matplotlib.pyplot as plt
x, u = solve_poisson(0.2)
print(f"Максимальная ошибка в центрах ячеек: {np.max(np.abs(u - u_ref(x))):.8f}")
x_ref = np.linspace(0, Lx, 300)
plt.plot(x_ref, u_ref(x_ref), label="Точное решение")
plt.scatter(x, u, label="МКО (центры ячеек)", color="red", s=16)
plt.scatter([0, Lx], [u_left, u_right], marker="s", label="Граничные условия")
plt.xlabel("x")
plt.ylabel("u")
plt.legend()
plt.grid()
plt.show()

In [ ]:
steps = 0.1 / 2**np.arange(8)
errors = []
print("    h          Макс. ошибка     Порядок")
for h in steps:
    x, u = solve_poisson(h)
    error = np.max(np.abs(u - u_ref(x)))
    order = f"{np.log2(errors[-1] / error):.3f}" if errors else "—"
    print(f"{h:10.8f}     {error:.4e}       {order}")
    errors.append(error)

plt.loglog(steps, errors, "ro-", label="Ошибка МКО")
plt.loglog(steps, errors[0] * (steps / steps[0])**2, "k--", label=r"$C h^2$")
plt.xlabel("Шаг сетки h")
plt.ylabel("Максимальная ошибка в центрах ячеек")
plt.grid(True, which="both")
plt.legend()
plt.show()